# Model Context Protocol (MCP) with LangGraph Agents

This notebook demonstrates how to create a LangGraph agent that utilizes the Model Context Protocol (MCP). MCP is an open-source standard launched by Anthropic in November 2024 that enables AI systems to seamlessly connect with external tools and data sources through a universal interface. The protocol has been adopted by major AI providers including OpenAI and Google DeepMind.

The following cell shows our imports and initializations. A new guest is `langchain_mcp_adapters`, which provides the `MultiServerMCPClient` to connect to multiple MCP servers. We realize that LangChain is ready for production use when we can use it with MCP.

In [ ]:
import os
import dotenv
from langchain.chat_models import init_chat_model
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.tools import load_mcp_tools
from langgraph.prebuilt import create_react_agent
from langfuse.langchain import CallbackHandler

# Load environment variables from .env file.
dotenv.load_dotenv()

# Initialize the Langfuse handler
langfuse_handler = CallbackHandler()

# Initialize the chat model.
model = init_chat_model(os.environ["LANGCHAIN_CHAT_MODEL_ANTHROPIC"])

Next we will create an MCP client that connects to two MCP servers. The first server `playwright` provides web browsing capabilities, while the second server `memory` offers access to a persistent memory store. We will also define a local file `memory.json` to store the agent's memory.

In [ ]:
# Define the path to the memory file.
import ipynbname
notebook_path = os.path.dirname(ipynbname.path())
memory_file_path = os.path.join(notebook_path, "memory.json")
print(f"Memory file path: {memory_file_path}")

# Delete the file if it exists.
if os.path.exists(memory_file_path):
    os.remove(memory_file_path)

# Create a multi-server MCP client with custom configurations.
client = MultiServerMCPClient(
    {
        "playwright": {
            "command": "npx",
            "transport": "stdio",
            "args": [
                "@playwright/mcp@latest"
            ]
        },
        "memory": {
            "command": "npx",
            "transport": "stdio",
            "args": [
                "-y",
                "@modelcontextprotocol/server-memory"
            ],
            "env": {
                "MEMORY_FILE_PATH": memory_file_path,
            }
        }
    }
)

## Running the Agent

As we remember, it is easy to integrate tools into LangGraph agentic workflows and agents. MCP is no exception, as the MCP tools will be mapped automatically to LangChain tools. We can then invoke the agent with a user prompt, and it will decide which tools to use based on the context.

First, let us find out which tools are available from the MCP client:

In [ ]:
# Get the tools.
print("Fetching tools from the MCP client...")
tools = await client.get_tools()
for tool in tools:
    print(f"Tool: {tool.name} - {tool.description}")

This is a fine collection of tools. LangGraph's ReAct agent will be able to use them as needed.

In [ ]:
async with client.session("playwright") as playwright_session, client.session("memory") as memory_session:
    playwright_tools = await load_mcp_tools(playwright_session)
    memory_tools = await load_mcp_tools(memory_session)
    tools = playwright_tools + memory_tools

    # Create the react agent without any tools.
    agent = create_react_agent(
        model=model,
        tools=tools,
        prompt=prompt
    )

    # Invoke the agent. This can take a while...
    print("Invoking the agent...")
    result = await agent.ainvoke(
        { "messages": [
            {
                "role": "user",
                "content": "Go to https://en.wikipedia.org/wiki/Model_Context_Protocol, wait for the text to appear read first paragraph, and store all the relevant information in memory. Make sure to store memories regularly."
            }
        ]},
        config={"callbacks": [langfuse_handler], "recursion_limit": 50}
    )
    print("Agent response:")
    print(result["messages"][-1].content)

Do not forget to take a look at the memory file `memory.json` to see how the agent's memory is stored persistently.


# Done.